# CS60075 — NLP Assignment 2
## Task A:- Dataset and Preprocessing

**Sayan Das (25CS60R01) · Mridul Kumar (25CS60R40)**

This notebook covers **Task A** of the assignment:
- A.1 Extract 500 validation articles from training set
- A.2 Remove punctuations and non-ASCII characters
- A.3 Stopword removal
- A.4 Stemming / Lemmatization
- A.5 Additional preprocessing (number normalisation, sentence splitting, caching)

Outputs are saved to `cache/taskA_output.pkl` for `taskB.ipynb` to load.

## Setup: Imports and Configuration

In [ ]:
import math
import importlib.util
import os
import pickle
import random
import re
import string
import time
import unicodedata
from collections import Counter
from contextlib import nullcontext
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

SEED = 42

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [ ]:
@dataclass
class Config:
    train_csv: str = "/content/drive/MyDrive/NLP Assignment 2/train.csv"
    test_csv: str = "/content/drive/MyDrive/NLP Assignment 2/test.csv"
    train_subset_size: int = 5000
    val_size: int = 500
    min_doc_freq_ratio: float = 0.01
    hidden_dim: int = 300
    embed_dim: int = 300
    batch_size: int = 16
    epochs: int = 5
    learning_rate: float = 2e-3
    teacher_forcing_ratio: float = 1.0
    teacher_forcing_start: float = 1.0
    teacher_forcing_end: float = 0.6
    max_input_tokens: int = 600
    max_output_tokens: int = 25
    max_sentences: int = 40
    max_sentence_tokens: int = 40
    dropout: float = 0.3
    beam_width: int = 4
    length_penalty_alpha: float = 0.7
    glove_path: Optional[str] = "/content/drive/MyDrive/NLP Assignment 2/glove.6B.300d.txt"
    num_workers: int = 0
    cache_dir: str = "/content/drive/MyDrive/NLP Assignment 2/cache"
    use_preprocess_cache: bool = True
    use_vocab_cache: bool = True
    use_attention: bool = True
    tie_embeddings: bool = False
    use_mixed_precision: bool = True

CFG = Config()
print(asdict(CFG))

{'train_csv': '/content/drive/MyDrive/NLP Assignment 2/train.csv', 'test_csv': '/content/drive/MyDrive/NLP Assignment 2/test.csv', 'train_subset_size': 5000, 'val_size': 500, 'min_doc_freq_ratio': 0.01, 'hidden_dim': 300, 'embed_dim': 300, 'batch_size': 16, 'epochs': 5, 'learning_rate': 0.002, 'teacher_forcing_ratio': 1.0, 'teacher_forcing_start': 1.0, 'teacher_forcing_end': 0.6, 'max_input_tokens': 600, 'max_output_tokens': 25, 'max_sentences': 40, 'max_sentence_tokens': 40, 'dropout': 0.3, 'beam_width': 4, 'length_penalty_alpha': 0.7, 'glove_path': '/content/drive/MyDrive/NLP Assignment 2/glove.6B.300d.txt', 'num_workers': 0, 'cache_dir': '/content/drive/MyDrive/NLP Assignment 2/cache', 'use_preprocess_cache': True, 'use_vocab_cache': True, 'use_attention': True, 'tie_embeddings': False, 'use_mixed_precision': True}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
installed_packages = {
    "nltk": importlib.util.find_spec("nltk") is not None,
    "rouge_score": importlib.util.find_spec("rouge_score") is not None,
}
installed_packages

{'nltk': True, 'rouge_score': False}

In [ ]:
import nltk

def ensure_nltk_resource(resource_name: str, download_name: Optional[str] = None):
    try:
        nltk.data.find(resource_name)
    except LookupError:
        nltk.download(download_name or resource_name.split("/")[-1], quiet=True)

ensure_nltk_resource("tokenizers/punkt", "punkt")
ensure_nltk_resource("corpora/stopwords", "stopwords")
ensure_nltk_resource("corpora/wordnet", "wordnet")
ensure_nltk_resource("corpora/omw-1.4", "omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize

STOPWORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

## Load Raw Dataset

In [ ]:
train_df_full = pd.read_csv(CFG.train_csv)
test_df = pd.read_csv(CFG.test_csv)

print("train.csv shape:", train_df_full.shape)
print("test.csv shape:", test_df.shape)
display(train_df_full.head(2))

train.csv shape: (13879, 2)
test.csv shape: (100, 2)


,title,text
0,"Port St. Lucie, Florida","Port St. Lucie is a city in St. Lucie County, ..."
1,Dirty Dozen,"Dirty Dozen may refer to:\n\nBooks, film and t..."


## Task A.1 — Train / Validation / Test Split

Randomly sample 5,000 articles from `train.csv`. Hold out 500 as the validation/development set; use the remaining 4,500 for training.

In [ ]:
sampled_train = (
    train_df_full
    .sample(n=CFG.train_subset_size, random_state=SEED)
    .reset_index(drop=True)
)

val_indices = sampled_train.sample(n=CFG.val_size, random_state=SEED).index
val_df = sampled_train.loc[val_indices].reset_index(drop=True)
train_df = sampled_train.drop(index=val_indices).reset_index(drop=True)

print("sampled train subset:", sampled_train.shape)
print("train split:", train_df.shape)
print("validation split:", val_df.shape)
print("test split:", test_df.shape)

sampled train subset: (5000, 2)
train split: (4500, 2)
validation split: (500, 2)
test split: (100, 2)


## Task A.2, A.3, A.4, A.5 — Preprocessing Pipeline

Steps applied to every article body and title:
- **A.2** Unicode NFKD normalisation → non-ASCII removal → punctuation removal
- **A.3** Stopword removal (NLTK English stopwords; applied to body only, not titles)
- **A.4** WordNet lemmatisation via `nltk.stem.WordNetLemmatizer`
- **A.5 (extra)** Lowercasing · whitespace normalisation · number normalisation (`→ <num>`) · sentence-level splitting for hierarchical encoder · truncation (body ≤ 600 tokens, ≤ 40 sentences × 40 tokens) · preprocessing cache to disk (pickle)

In [ ]:
SENT_SPLIT_REGEX = re.compile(r"(?<=[.!?])\s+")
PUNCT_TABLE = str.maketrans("", "", string.punctuation)

def normalize_ascii(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text))
    text = text.encode("ascii", "ignore").decode("ascii")
    return text

def simple_sentence_split(text: str) -> List[str]:
    text = text.strip()
    if not text:
        return []
    try:
        return [s.strip() for s in sent_tokenize(text) if s.strip()]
    except LookupError:
        return [s.strip() for s in SENT_SPLIT_REGEX.split(text) if s.strip()]

def preprocess_text(
    text: str,
    remove_stopwords: bool = True,
    lemmatize: bool = True,
) -> List[str]:
    text = normalize_ascii(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\d+(?:[.,:/-]\d+)*", " <num> ", text)
    text = text.translate(PUNCT_TABLE)
    raw_tokens = text.split()
    tokens = []
    for tok in raw_tokens:
        if not tok:
            continue
        if remove_stopwords and tok in STOPWORDS:
            continue
        if lemmatize and tok not in {"<num>"}:
            tok = LEMMATIZER.lemmatize(tok)
        tokens.append(tok)
    return tokens

def preprocess_text_for_sentences(
    text: str,
    max_sentences: Optional[int] = None,
    max_sentence_tokens: Optional[int] = None,
) -> List[List[str]]:
    sentences = simple_sentence_split(normalize_ascii(text))
    processed = []
    for sentence in sentences:
        tokens = preprocess_text(sentence)
        if max_sentence_tokens is not None:
            tokens = tokens[:max_sentence_tokens]
        if tokens:
            processed.append(tokens)
    if max_sentences is not None:
        processed = processed[:max_sentences]
    return processed

In [ ]:
def attach_processed_columns(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="preprocess"):
        title_tokens = preprocess_text(row["title"], remove_stopwords=False, lemmatize=True)
        text_tokens = preprocess_text(row["text"], remove_stopwords=True, lemmatize=True)[: cfg.max_input_tokens]
        sentence_tokens = preprocess_text_for_sentences(
            row["text"],
            max_sentences=cfg.max_sentences,
            max_sentence_tokens=cfg.max_sentence_tokens,
        )
        flat_sentence_tokens = [tok for sent in sentence_tokens for tok in sent][: cfg.max_input_tokens]
        rows.append(
            {
                "title": row["title"],
                "text": row["text"],
                "title_tokens": title_tokens,
                "text_tokens": text_tokens,
                "sentence_tokens": sentence_tokens,
                "flat_sentence_tokens": flat_sentence_tokens,
            }
        )
    return pd.DataFrame(rows)

def get_preprocess_cache_path(cfg: Config) -> Path:
    cache_dir = Path(cfg.cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_name = (
        f"prep_train{cfg.train_subset_size}_val{cfg.val_size}"
        f"_in{cfg.max_input_tokens}_sent{cfg.max_sentences}_tok{cfg.max_sentence_tokens}"
        f"_seed{SEED}.pkl"
    )
    return cache_dir / cache_name

preprocess_cache_path = get_preprocess_cache_path(CFG)
if CFG.use_preprocess_cache and preprocess_cache_path.exists():
    with preprocess_cache_path.open("rb") as f:
        cached = pickle.load(f)
    train_proc = cached["train_proc"]
    val_proc = cached["val_proc"]
    test_proc = cached["test_proc"]
    print(f"Loaded cached preprocessing from {preprocess_cache_path}")
else:
    train_proc = attach_processed_columns(train_df, CFG)
    val_proc = attach_processed_columns(val_df, CFG)
    test_proc = attach_processed_columns(test_df, CFG)
    if CFG.use_preprocess_cache:
        with preprocess_cache_path.open("wb") as f:
            pickle.dump(
                {
                    "train_proc": train_proc,
                    "val_proc": val_proc,
                    "test_proc": test_proc,
                },
                f,
            )
        print(f"Saved preprocessing cache to {preprocess_cache_path}")

display(train_proc[["title", "title_tokens", "text_tokens"]].head(2))

Loaded cached preprocessing from /content/drive/MyDrive/NLP Assignment 2/cache/prep_train5000_val500_in600_sent40_tok40_seed42.pkl


,title,title_tokens,text_tokens
0,Oklahoma Panhandle State University,"[oklahoma, panhandle, state, university]","[oklahoma, panhandle, state, university, opsu,..."
1,Suslov,[suslov],"[suslov, russian, masculine, surname, feminine..."


## Task A — Vocabulary Construction

Build a shared encoder–decoder vocabulary from the training split. Only tokens appearing in **≥ 1% of training documents** (≥ 45 docs) are retained. Special tokens `<pad>`, `<unk>`, `<bos>`, `<eos>` are always included.

In [ ]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
NUM_TOKEN = "<num>"
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

class Vocab:
    def __init__(self, stoi: Dict[str, int], itos: List[str]):
        self.stoi = stoi
        self.itos = itos
        self.pad_id = stoi[PAD_TOKEN]
        self.unk_id = stoi[UNK_TOKEN]
        self.bos_id = stoi[BOS_TOKEN]
        self.eos_id = stoi[EOS_TOKEN]

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens: Sequence[str]) -> List[int]:
        return [self.stoi.get(tok, self.unk_id) for tok in tokens]

    def decode(self, ids: Sequence[int], skip_special: bool = True) -> List[str]:
        tokens = []
        for idx in ids:
            token = self.itos[int(idx)]
            if skip_special and token in SPECIAL_TOKENS:
                continue
            tokens.append(token)
        return tokens

def build_vocab(train_frame: pd.DataFrame, min_doc_freq_ratio: float = 0.01) -> Vocab:
    min_df = max(1, math.ceil(min_doc_freq_ratio * len(train_frame)))
    doc_freq = Counter()
    for _, row in train_frame.iterrows():
        unique_tokens = set(row["text_tokens"]) | set(row["title_tokens"])
        doc_freq.update(unique_tokens)
    tokens = [tok for tok, freq in doc_freq.items() if freq >= min_df]
    tokens = sorted(tokens)
    itos = SPECIAL_TOKENS + tokens
    stoi = {tok: idx for idx, tok in enumerate(itos)}
    return Vocab(stoi, itos)

def get_vocab_cache_path(cfg: Config) -> Path:
    cache_dir = Path(cfg.cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_name = (
        f"vocab_train{cfg.train_subset_size}_df{cfg.min_doc_freq_ratio}"
        f"_seed{SEED}.pkl"
    )
    return cache_dir / cache_name

vocab_cache_path = get_vocab_cache_path(CFG)
if CFG.use_vocab_cache and vocab_cache_path.exists():
    with vocab_cache_path.open("rb") as f:
        vocab_payload = pickle.load(f)
    vocab = Vocab(vocab_payload["stoi"], vocab_payload["itos"])
    print(f"Loaded cached vocabulary from {vocab_cache_path}")
else:
    vocab = build_vocab(train_proc, CFG.min_doc_freq_ratio)
    if CFG.use_vocab_cache:
        with vocab_cache_path.open("wb") as f:
            pickle.dump({"stoi": vocab.stoi, "itos": vocab.itos}, f)
        print(f"Saved vocabulary cache to {vocab_cache_path}")

print("vocab size:", len(vocab))
print("sample tokens:", vocab.itos[:20])

Loaded cached vocabulary from /content/drive/MyDrive/NLP Assignment 2/cache/vocab_train5000_df0.01_seed42.pkl
vocab size: 4193
sample tokens: ['<pad>', '<unk>', '<bos>', '<eos>', 'aaron', 'abandoned', 'abbey', 'abbreviated', 'abc', 'aberdeen', 'ability', 'able', 'aboard', 'abolished', 'abraham', 'abroad', 'absence', 'absorbed', 'abuse', 'ac']


## Task A — Dataset and DataLoader

Wrap the processed data in a PyTorch `Dataset` and create `DataLoader` objects for train, validation, and test splits.

In [ ]:
class TitleDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, vocab: Vocab):
        self.frame = frame.reset_index(drop=True)
        self.vocab = vocab

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        src_ids = self.vocab.encode(row["text_tokens"])
        tgt_ids = [self.vocab.bos_id] + self.vocab.encode(row["title_tokens"]) + [self.vocab.eos_id]

        sentence_ids = []
        sentence_lengths = []
        for sent in row["sentence_tokens"]:
            if sent:
                encoded = self.vocab.encode(sent)
                sentence_ids.extend(encoded)
                sentence_lengths.append(len(encoded))

        if not sentence_ids:
            sentence_ids = src_ids[:]
            sentence_lengths = [len(src_ids)] if src_ids else [1]

        return {
            "src_ids": torch.tensor(src_ids or [self.vocab.unk_id], dtype=torch.long),
            "tgt_ids": torch.tensor(tgt_ids, dtype=torch.long),
            "hier_src_ids": torch.tensor(sentence_ids or [self.vocab.unk_id], dtype=torch.long),
            "sentence_lengths": torch.tensor(sentence_lengths, dtype=torch.long),
            "raw_title": row["title"],
        }

def collate_batch(batch):
    src = [item["src_ids"] for item in batch]
    tgt = [item["tgt_ids"] for item in batch]
    hier_src = [item["hier_src_ids"] for item in batch]
    sentence_lengths = [item["sentence_lengths"] for item in batch]

    src_lengths = torch.tensor([len(x) for x in src], dtype=torch.long)
    tgt_lengths = torch.tensor([len(x) for x in tgt], dtype=torch.long)
    hier_lengths = torch.tensor([len(x) for x in hier_src], dtype=torch.long)
    num_sentences = torch.tensor([len(x) for x in sentence_lengths], dtype=torch.long)

    src_padded = pad_sequence(src, batch_first=True, padding_value=vocab.pad_id)
    tgt_padded = pad_sequence(tgt, batch_first=True, padding_value=vocab.pad_id)
    hier_padded = pad_sequence(hier_src, batch_first=True, padding_value=vocab.pad_id)
    sentence_lengths_padded = pad_sequence(sentence_lengths, batch_first=True, padding_value=0)

    return {
        "src_ids": src_padded,
        "src_lengths": src_lengths,
        "tgt_ids": tgt_padded,
        "tgt_lengths": tgt_lengths,
        "hier_src_ids": hier_padded,
        "hier_lengths": hier_lengths,
        "sentence_lengths": sentence_lengths_padded,
        "num_sentences": num_sentences,
        "raw_titles": [item["raw_title"] for item in batch],
    }

train_ds = TitleDataset(train_proc, vocab)
val_ds = TitleDataset(val_proc, vocab)
test_ds = TitleDataset(test_proc, vocab)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    collate_fn=collate_batch,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_batch,
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_batch,
)

first_batch = next(iter(train_loader))
{k: tuple(v.shape) if torch.is_tensor(v) else len(v) for k, v in first_batch.items()}

{'src_ids': (16, 600),
 'src_lengths': (16,),
 'tgt_ids': (16, 6),
 'tgt_lengths': (16,),
 'hier_src_ids': (16, 568),
 'hier_lengths': (16,),
 'sentence_lengths': (16, 40),
 'num_sentences': (16,),
 'raw_titles': 16}

## Save Task A Outputs for taskB.ipynb

In [ ]:
# ── Save Task A outputs so taskB.ipynb can load them ──────────────────────────
import pickle, os

os.makedirs(CFG.cache_dir, exist_ok=True)

with open(os.path.join(CFG.cache_dir, "taskA_output.pkl"), "wb") as f:
    pickle.dump({
        "train_proc": train_proc,
        "val_proc":   val_proc,
        "test_proc":  test_proc,
        "vocab_stoi": vocab.stoi,
        "vocab_itos": vocab.itos,
    }, f)

print(f"Task A complete. Saved to {CFG.cache_dir}/taskA_output.pkl")
print(f"Vocab size      : {len(vocab)}")
print(f"Train examples  : {len(train_proc)}")
print(f"Val examples    : {len(val_proc)}")
print(f"Test examples   : {len(test_proc)}")

Task A complete. Saved to /content/drive/MyDrive/NLP Assignment 2/cache/taskA_output.pkl
Vocab size      : 4193
Train examples  : 4500
Val examples    : 500
Test examples   : 100
